In [1]:
%pip install textstat vaderSentiment spacy textblob imbalanced-learn xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.1/239.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 46.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics.pairwise import cosine_similarity
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from textstat import flesch_reading_ease
import matplotlib.pyplot as plt
import seaborn as sns
import uuid
import spacy
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import joblib
import os

In [3]:
# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

class DebateAnalyzer:
    def __init__(self, model_type='random_forest', is_speaker_level=False):
        self.model_type = model_type
        self.is_speaker_level = is_speaker_level
        self.model = None
        self.calibrated_model = None  # For probability calibration
        self.vectorizer = TfidfVectorizer(max_features=2000, stop_words='english')  # Increased max_features
        self.scaler = StandardScaler()
        self.feature_names = None
        self.fitted_vectorizer = None
        self.fitted_scaler = None
        self.nlp = spacy.load('en_core_web_sm')  # For embeddings
        self.sentiment_analyzer = SentimentIntensityAnalyzer()

    def train(self, X, y):
        # Handle imbalance with SMOTE
        smote = SMOTE(random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X, y)

        if self.model_type == 'xgboost':
            self.model = XGBClassifier(random_state=42)
            params = {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.01, 0.1]}
            grid = GridSearchCV(self.model, params, cv=5)
            grid.fit(X_resampled, y_resampled)
            self.model = grid.best_estimator_
        elif self.model_type == 'ensemble':
            rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
            gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
            lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
            self.model = VotingClassifier(estimators=[('rf', rf), ('gb', gb), ('lr', lr)], voting='soft')
            self.model.fit(X_resampled, y_resampled)
        elif self.model_type == 'random_forest':
            self.model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', max_depth=10)
            self.model.fit(X_resampled, y_resampled)
        elif self.model_type == 'gradient_boosting':
            self.model = GradientBoostingClassifier(n_estimators=100, random_state=42)
            self.model.fit(X_resampled, y_resampled)
        elif self.model_type == 'logistic_regression':
            self.model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
            self.model.fit(X_resampled, y_resampled)
        else:
            raise ValueError("Unsupported model type")

        # Calibrate probabilities
        self.calibrated_model = CalibratedClassifierCV(self.model, method='sigmoid', cv=3)
        self.calibrated_model.fit(X_resampled, y_resampled)

    def predict_winner(self, new_argument, is_for_position=True, opponent_text=None):
        if self.model is None or self.fitted_vectorizer is None or self.fitted_scaler is None:
            raise ValueError("Model, vectorizer, or scaler not trained/fitted yet")

        # Use the appropriate feature preparation method
        if self.is_speaker_level:
            new_features = self.prepare_speaker_features_for_prediction(new_argument, is_for_position, opponent_text)
        else:
            new_features = self.prepare_segment_features(new_argument, is_for_position, opponent_text)

        new_features_scaled = self.fitted_scaler.transform(new_features)
        prediction = self.calibrated_model.predict(new_features_scaled)[0]  # Use calibrated model
        probabilities = self.calibrated_model.predict_proba(new_features_scaled)[0]

        return {
            'prediction': 'YES' if prediction == 1 else 'NO',
            'probability': probabilities[1],
            'confidence': max(probabilities)
        }

    def prepare_segment_features(self, text, is_for_position, opponent_text=None):
        # Tokenize and process text
        tokens = word_tokenize(text.lower())
        stop_words = set(stopwords.words('english'))
        filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]

        # Linguistic features
        word_count = len(filtered_tokens)
        unique_words = len(set(filtered_tokens))
        evidence_words = sum(1 for word in filtered_tokens if word in ['evidence', 'study', 'data', 'research', 'fact'])
        strong_words = sum(1 for word in filtered_tokens if word in ['must', 'should', 'always', 'never'])
        reading_ease = flesch_reading_ease(text)

        # New: Sentiment features
        blob = TextBlob(text)
        sentiment = self.sentiment_analyzer.polarity_scores(text)
        polarity = blob.sentiment.polarity
        subjectivity = blob.sentiment.subjectivity

        # New: SpaCy embeddings (average vector, truncate to 50 dims)
        doc = self.nlp(text)
        embedding = doc.vector[:50]

        # New: Rebuttal strength
        rebuttal_sim = 0
        negation_count = sum(1 for token in tokens if token in ['not', 'no', 'never', 'against'])
        if opponent_text:
            opp_vec = self.fitted_vectorizer.transform([opponent_text]).toarray()
            text_vec = self.fitted_vectorizer.transform([text]).toarray()
            rebuttal_sim = cosine_similarity(text_vec, opp_vec)[0][0]

        # TF-IDF features
        tfidf_features = self.fitted_vectorizer.transform([text]).toarray()

        # Combine features
        features = np.hstack([
            tfidf_features,
            [[
                1 if is_for_position else 0,
                word_count,
                unique_words / word_count if word_count > 0 else 0,
                evidence_words,
                strong_words,
                reading_ease,
                len(text),
                polarity, subjectivity, sentiment['compound'],  # Sentiment
                rebuttal_sim, negation_count  # Rebuttal
            ]],
            [embedding]  # Embeddings
        ])

        return features

    def prepare_speaker_features_for_prediction(self, text, is_for_position, opponent_text=None):
        # Tokenize and process text
        tokens = word_tokenize(text.lower())
        stop_words = set(stopwords.words('english'))
        filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]

        # Linguistic features
        word_count = len(filtered_tokens)
        unique_words = len(set(filtered_tokens))
        evidence_words = sum(1 for word in filtered_tokens if word in ['evidence', 'study', 'data', 'research', 'fact'])
        strong_words = sum(1 for word in filtered_tokens if word in ['must', 'should', 'always', 'never'])
        reading_ease = flesch_reading_ease(text)
        num_segments = 1  # Assume single segment for prediction
        avg_words_per_segment = word_count / num_segments

        # New: Sentiment features
        blob = TextBlob(text)
        sentiment = self.sentiment_analyzer.polarity_scores(text)
        polarity = blob.sentiment.polarity
        subjectivity = blob.sentiment.subjectivity

        # New: SpaCy embeddings (average vector, truncate to 50 dims)
        doc = self.nlp(text)
        embedding = doc.vector[:50]

        # New: Rebuttal strength
        rebuttal_sim = 0
        negation_count = sum(1 for token in tokens if token in ['not', 'no', 'never', 'against'])
        if opponent_text:
            opp_vec = self.fitted_vectorizer.transform([opponent_text]).toarray()
            text_vec = self.fitted_vectorizer.transform([text]).toarray()
            rebuttal_sim = cosine_similarity(text_vec, opp_vec)[0][0]

        # TF-IDF features
        tfidf_features = self.fitted_vectorizer.transform([text]).toarray()

        # Combine features
        features = np.hstack([
            tfidf_features,
            [[
                1 if is_for_position else 0,
                word_count,
                unique_words / word_count if word_count > 0 else 0,
                evidence_words,
                strong_words,
                reading_ease,
                len(text),
                num_segments,
                avg_words_per_segment,
                polarity, subjectivity, sentiment['compound'],  # Sentiment
                rebuttal_sim, negation_count  # Rebuttal
            ]],
            [embedding]  # Embeddings
        ])

        return features

    def get_feature_importance(self):
        if self.model_type in ['random_forest', 'gradient_boosting', 'xgboost'] and self.model is not None:
            importances = self.model.feature_importances_
            feature_importance = pd.DataFrame({
                'feature': self.feature_names,
                'importance': importances
            }).sort_values('importance', ascending=False)
            return feature_importance
        return None
    def save_model(self, directory='trained_model'):
        """Saves the trained model and its components to a directory."""
        if not os.path.exists(directory):
            os.makedirs(directory)

        if self.model:
            joblib.dump(self.model, os.path.join(directory, 'model.joblib'))
        if self.calibrated_model:
            joblib.dump(self.calibrated_model, os.path.join(directory, 'calibrated_model.joblib'))
        if self.fitted_vectorizer:
            joblib.dump(self.fitted_vectorizer, os.path.join(directory, 'vectorizer.joblib'))
        if self.fitted_scaler:
            joblib.dump(self.fitted_scaler, os.path.join(directory, 'scaler.joblib'))
        if self.feature_names:
            joblib.dump(self.feature_names, os.path.join(directory, 'feature_names.joblib'))
        print(f"Model and components saved to '{directory}' directory.")

    def load_model(self, directory='trained_model'):
        """Loads a trained model and its components from a directory."""
        self.model = joblib.load(os.path.join(directory, 'model.joblib'))
        self.calibrated_model = joblib.load(os.path.join(directory, 'calibrated_model.joblib'))
        self.fitted_vectorizer = joblib.load(os.path.join(directory, 'vectorizer.joblib'))
        self.fitted_scaler = joblib.load(os.path.join(directory, 'scaler.joblib'))
        self.feature_names = joblib.load(os.path.join(directory, 'feature_names.joblib'))
        # Set the vectorizer and scaler on the class instance after loading
        self.vectorizer = self.fitted_vectorizer
        self.scaler = self.fitted_scaler
        print(f"Model and components loaded from '{directory}' directory.")

def load_and_prepare_data(file_path):
    print("Loading Intelligence Squared debate data...")
    df = pd.read_csv(file_path)

    # Rename 'text' to 'argument_text'
    df = df.rename(columns={'text': 'argument_text'})

    # Drop rows with missing 'argument_text'
    initial_rows = len(df)
    df.dropna(subset=['argument_text'], inplace=True)
    rows_dropped = initial_rows - len(df)
    if rows_dropped > 0:
        print(f"Dropped {rows_dropped} rows with missing 'argument_text'.")

    # Create winner column
    df['winner'] = df.apply(
        lambda row: 1 if row['speaker_name'] == row['conversation_winner'] and row['conversation_winner'] != "It's a tie!" else 0,
        axis=1
    )

    # Basic dataset overview
    print("\nDataset Overview:")
    print(f"- Total segments: {len(df)}")
    print(f"- Unique debates: {df['conversation_id'].nunique()}")
    print(f"- Unique speakers: {df['speaker_name'].nunique()}")
    print(f"- Winner distribution:\n{df['winner'].value_counts()}")
    print(f"- Position distribution:\n{df['speakertype'].value_counts()}")

    return df

def analyze_debate_dynamics(df):
    print("\n==================================================")
    print("DEBATE DYNAMICS ANALYSIS")
    print("==================================================")

    # Speaking patterns by position and outcome
    patterns = df.groupby(['speakertype', 'winner']).agg({
        'argument_text': [
            ('word_count', lambda x: x.str.split().str.len().mean()),
            ('std', lambda x: x.str.split().str.len().std()),
            ('sum', lambda x: x.str.split().str.len().sum())
        ],
        'argument_text': [
            ('char_count', lambda x: x.str.len().mean()),
            ('std', lambda x: x.str.len().std())
        ],
        'conversation_id': 'count'
    }).round(2)

    print("\nSpeaking patterns by position and outcome:")
    print(patterns)

    # Speaker performance summary
    speaker_summary = df.groupby('speaker_name').agg({
        'winner': 'sum',
        'argument_text': [
            ('word_count', lambda x: x.str.split().str.len().mean()),
            ('count', 'count')
        ],
        'speakertype': 'first',
        'conversation_title': 'first'
    }).round(2)

    print("\nSpeaker Performance Summary:")
    print(speaker_summary)

    # Topic-wise results
    print("\nTopic-wise Results:")
    for topic in df['conversation_title'].unique():
        topic_df = df[df['conversation_title'] == topic]
        print(f"\n{topic}:")
        for _, row in topic_df.groupby('speaker_name').agg({
            'argument_text': lambda x: x.str.split().str.len().sum(),
            'winner': 'first',
            'speakertype': 'first'
        }).iterrows():
            print(f"  {row['speakertype'].upper()}: {row.name} ({row['argument_text']} words) - {'WON' if row['winner'] == 1 else 'LOST'}")

def prepare_segment_features(df, vectorizer):
    print("Extracting features for individual segments...")

    # Ensure 'argument_text' is string type
    df['argument_text'] = df['argument_text'].astype(str).fillna('')

    # TF-IDF features
    X_tfidf = vectorizer.fit_transform(df['argument_text']).toarray()

    # Linguistic and metadata features
    features = []
    for _, row in df.iterrows():
        text = row['argument_text']
        tokens = word_tokenize(text.lower())
        stop_words = set(stopwords.words('english'))
        filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]

        word_count = len(filtered_tokens)
        unique_words = len(set(filtered_tokens))
        evidence_words = sum(1 for word in filtered_tokens if word in ['evidence', 'study', 'data', 'research', 'fact'])
        strong_words = sum(1 for word in filtered_tokens if word in ['must', 'should', 'always', 'never'])
        reading_ease = flesch_reading_ease(text)

        features.append([
            1 if row['speakertype'] == 'for' else 0,
            word_count,
            unique_words / word_count if word_count > 0 else 0,
            evidence_words,
            strong_words,
            reading_ease,
            len(text)
        ])

    X_features = np.array(features)
    X = np.hstack([X_tfidf, X_features])

    feature_names = (
        [f'tfidf_{i}' for i in range(X_tfidf.shape[1])] +
        ['is_for_position', 'word_count', 'unique_word_ratio', 'evidence_words_count', 'strong_words_count', 'reading_ease', 'segment_length']
    )

    print(f"Segment feature matrix shape: {X.shape}")
    print(f"Winner distribution:\n{df['winner'].value_counts()}")

    return X, df['winner'].values, feature_names

def prepare_speaker_features(df, vectorizer):
    print("Extracting features for complete speaker arguments...")

    # Aggregate by conversation_id and speaker_name
    speaker_df = df.groupby(['conversation_id', 'speaker_name']).agg({
        'argument_text': lambda x: ' '.join(x.astype(str)),
        'winner': 'first',
        'speakertype': 'first',
        'conversation_title': 'first'
    }).reset_index()

    # Ensure 'argument_text' is string type
    speaker_df['argument_text'] = speaker_df['argument_text'].astype(str).fillna('')

    # TF-IDF features
    X_tfidf = vectorizer.fit_transform(speaker_df['argument_text']).toarray()

    # Speaker-level features
    features = []
    embeddings_list = []  # Collect embeddings separately
    nlp = spacy.load('en_core_web_sm')
    sentiment_analyzer = SentimentIntensityAnalyzer()

    for _, row in speaker_df.iterrows():
        text = row['argument_text']
        tokens = word_tokenize(text.lower())
        stop_words = set(stopwords.words('english'))
        filtered_tokens = [word for word in tokens if word.isalnum() and word not in stop_words]

        word_count = len(filtered_tokens)
        unique_words = len(set(filtered_tokens))
        evidence_words = sum(1 for word in filtered_tokens if word in ['evidence', 'study', 'data', 'research', 'fact'])
        strong_words = sum(1 for word in filtered_tokens if word in ['must', 'should', 'always', 'never'])
        reading_ease = flesch_reading_ease(text)
        num_segments = df[(df['conversation_id'] == row['conversation_id']) &
                          (df['speaker_name'] == row['speaker_name'])].shape[0]
        avg_words_per_segment = word_count / num_segments if num_segments > 0 else 0

        # Sentiment features
        blob = TextBlob(text)
        sentiment = sentiment_analyzer.polarity_scores(text)
        polarity = blob.sentiment.polarity
        subjectivity = blob.sentiment.subjectivity
        compound = sentiment['compound']

        # Rebuttal strength
        negation_count = sum(1 for token in tokens if token in ['not', 'no', 'never', 'against'])
        rebuttal_sim = 0
        # Find opponent in the same conversation
        opponent_df = speaker_df[(speaker_df['conversation_id'] == row['conversation_id']) &
                                 (speaker_df['speaker_name'] != row['speaker_name'])]
        if not opponent_df.empty:
            opponent_text = opponent_df['argument_text'].iloc[0]
            opp_vec = vectorizer.transform([opponent_text]).toarray()
            text_vec = vectorizer.transform([text]).toarray()
            rebuttal_sim = cosine_similarity(text_vec, opp_vec)[0][0]

        # SpaCy embeddings (average vector, truncate to 50 dims)
        doc = nlp(text)
        embedding = doc.vector[:50]
        embeddings_list.append(embedding)

        features.append([
            1 if row['speakertype'] == 'for' else 0,
            word_count,
            unique_words / word_count if word_count > 0 else 0,
            evidence_words,
            strong_words,
            reading_ease,
            len(text),
            num_segments,
            avg_words_per_segment,
            polarity, subjectivity, compound,  # Sentiment
            rebuttal_sim, negation_count  # Rebuttal
        ])

    X_features = np.array(features)
    X_embeddings = np.array(embeddings_list)
    X = np.hstack([X_tfidf, X_features, X_embeddings])

    feature_names = (
        [f'tfidf_{i}' for i in range(X_tfidf.shape[1])] +
        ['is_for_position', 'total_words', 'unique_word_ratio', 'evidence_words_count',
         'strong_words_count', 'reading_ease', 'text_length', 'num_segments', 'avg_words_per_segment',
         'polarity', 'subjectivity', 'compound', 'rebuttal_sim', 'negation_count'] +
        [f'embedding_{i}' for i in range(50)]
    )

    print(f"Speaker feature matrix shape: {X.shape}")
    print(f"Number of speakers: {len(speaker_df)}")
    print(f"Winner distribution:\n{speaker_df['winner'].value_counts()}")

    return X, speaker_df['winner'].values, feature_names, speaker_df

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
# Load data
df = load_and_prepare_data('debate.csv')

# Analyze debate dynamics
analyze_debate_dynamics(df)

# # Segment-level prediction
# print("\n============================================================")
# print("APPROACH 1: SEGMENT-LEVEL PREDICTION")
# print("(Predicting winner for individual argument segments)")
# print("============================================================")

# segment_analyzer = DebateAnalyzer(model_type='random_forest', is_speaker_level=False)
# X_segments, y_segments, segment_feature_names = prepare_segment_features(df, segment_analyzer.vectorizer)

# # Fit and store scaler and vectorizer
# X_segments_scaled = segment_analyzer.scaler.fit_transform(X_segments)
# segment_analyzer.fitted_scaler = segment_analyzer.scaler
# segment_analyzer.fitted_vectorizer = segment_analyzer.vectorizer
# segment_analyzer.feature_names = segment_feature_names

# # Train and evaluate
# X_train, X_test, y_train, y_test = train_test_split(X_segments_scaled, y_segments, test_size=0.2, random_state=42)
# segment_analyzer.train(X_train, y_train)

# print("\n--- SEGMENT MODEL: RANDOM FOREST ---")
# y_pred = segment_analyzer.model.predict(X_test)
# print(f"Model: random_forest")
# print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
# print("Classification Report:")
# print(classification_report(y_test, y_pred))

# # Feature importance
# feature_importance = segment_analyzer.get_feature_importance()
# if feature_importance is not None:
#     print("\n--- SEGMENT FEATURE IMPORTANCE ---")
#     print("Top 15 Most Important Features:")
#     print(feature_importance.head(15))

#     plt.figure(figsize=(10, 6))
#     sns.barplot(x='importance', y='feature', data=feature_importance.head(15))
#     plt.title('Top 15 Feature Importance (Segment-Level)')
#     plt.show()

# # Try other segment-level models
# for model_type in ['gradient_boosting', 'logistic_regression']:
#     print(f"\n--- SEGMENT MODEL: {model_type.upper()} ---")
#     temp_analyzer = DebateAnalyzer(model_type=model_type, is_speaker_level=False)
#     temp_analyzer.fitted_vectorizer = segment_analyzer.fitted_vectorizer
#     temp_analyzer.fitted_scaler = segment_analyzer.fitted_scaler
#     temp_analyzer.feature_names = segment_feature_names
#     X_segments_scaled = temp_analyzer.fitted_scaler.transform(X_segments)
#     X_train, X_test, y_train, y_test = train_test_split(X_segments_scaled, y_segments, test_size=0.2, random_state=42)
#     temp_analyzer.train(X_train, y_train)
#     y_pred = temp_analyzer.model.predict(X_test)
#     print(f"Model: {model_type}")
#     print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
#     print("Classification Report:")
#     print(classification_report(y_test, y_pred))

# Speaker-level prediction
print("\n============================================================")
print("APPROACH 2: SPEAKER-LEVEL PREDICTION")
print("(Predicting winner based on complete speaker arguments)")
print("============================================================")

speaker_analyzer = DebateAnalyzer(model_type='ensemble', is_speaker_level=True)  # Or 'xgboost'
X_speakers, y_speakers, speaker_feature_names, speaker_df = prepare_speaker_features(df, speaker_analyzer.vectorizer)  # Update prepare_speaker_features similarly if needed

# Fit scaler and vectorizer
X_speakers_scaled = speaker_analyzer.scaler.fit_transform(X_speakers)
speaker_analyzer.fitted_scaler = speaker_analyzer.scaler
speaker_analyzer.fitted_vectorizer = speaker_analyzer.vectorizer
speaker_analyzer.feature_names = speaker_feature_names  # Update feature_names to include new ones

# Train
X_train, X_test, y_train, y_test = train_test_split(X_speakers_scaled, y_speakers, test_size=0.2, random_state=42)
speaker_analyzer.train(X_train, y_train)

print("\n--- SPEAKER MODEL: RANDOM FOREST ---")
y_pred = speaker_analyzer.model.predict(X_test)
print(f"Model: random_forest")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Feature importance
feature_importance = speaker_analyzer.get_feature_importance()
if feature_importance is not None:
    print("\n--- SPEAKER FEATURE IMPORTANCE ---")
    print("Top 15 Most Important Features:")
    print(feature_importance.head(15))

    plt.figure(figsize=(10, 6))
    sns.barplot(x='importance', y='feature', data=feature_importance.head(15))
    plt.title('Top 15 Feature Importance (Speaker-Level)')
    plt.show()

# Try other speaker-level models
for model_type in ['gradient_boosting', 'logistic_regression']:
    print(f"\n--- SPEAKER MODEL: {model_type.upper()} ---")
    temp_analyzer = DebateAnalyzer(model_type=model_type, is_speaker_level=True)
    temp_analyzer.fitted_vectorizer = speaker_analyzer.fitted_vectorizer
    temp_analyzer.fitted_scaler = speaker_analyzer.fitted_scaler
    temp_analyzer.feature_names = speaker_feature_names
    X_speakers_scaled = temp_analyzer.fitted_scaler.transform(X_speakers)
    X_train, X_test, y_train, y_test = train_test_split(X_speakers_scaled, y_speakers, test_size=0.2, random_state=42)
    temp_analyzer.train(X_train, y_train)
    y_pred = temp_analyzer.model.predict(X_test)
    print(f"Model: {model_type}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))





speaker_analyzer.save_model(directory='my_debate_model')


# # Debate predictions
# print("\n============================================================")
# print("DEBATE PREDICTIONS")
# print("============================================================")

# for conv_id in df['conversation_id'].unique():
#     conv_df = df[df['conversation_id'] == conv_id]
#     speakers = conv_df['speaker_name'].unique()
#     conv_title = conv_df['conversation_title'].iloc[0]
#     ground_truth_winner = conv_df['conversation_winner'].iloc[0]

#     # Conversation history
#     history = []
#     for _, row in conv_df.iterrows():
#         speaker = row['speaker_name']
#         text = row['argument_text'].strip()
#         if len(text) > 100:
#             text = text[:97] + "..."
#         history.append(f"{speaker}: \"{text}\"")

#     # Aggregate text and positions
#     speaker_texts = {}
#     speaker_positions = {}
#     for speaker in speakers:
#         speaker_text = ' '.join(conv_df[conv_df['speaker_name'] == speaker]['argument_text'].astype(str))
#         speaker_position = conv_df[conv_df['speaker_name'] == speaker]['speakertype'].iloc[0]
#         speaker_texts[speaker] = speaker_text
#         speaker_positions[speaker] = speaker_position

#     # Predict for each speaker
#     results = {}
#     for speaker, text in speaker_texts.items():
#         is_for_position = (speaker_positions[speaker] == 'for')
#         result = speaker_analyzer.predict_winner(text, is_for_position)
#         results[speaker] = result['probability']

#     # Determine predicted winner
#     predicted_winner = max(results, key=results.get)
#     predicted_winner_prob = results[predicted_winner]

#     # Print results
#     print(f"\nConversation ID: {conv_id} ({conv_title})")
#     print("History = [")
#     for line in history:
#         print(f"  {line}")
#     print("]")
#     result = "Win" if ground_truth_winner != "It's a tie!" else "Tie"
#     print(f"Ground Truth: {ground_truth_winner} {result}")

#     print("\nSpeaker Predictions:")
#     for speaker, prob in results.items():
#         print(f"  {speaker}: {prob:.3f}")

#     print(f"\nJudgment: {predicted_winner} is predicted to WIN with {predicted_winner_prob:.3f} probability.")



Loading Intelligence Squared debate data...
Dropped 106 rows with missing 'argument_text'.

Dataset Overview:
- Total segments: 20103
- Unique debates: 1005
- Unique speakers: 826
- Winner distribution:
winner
0    13944
1     6159
Name: count, dtype: int64
- Position distribution:
speakertype
for        10148
against     9955
Name: count, dtype: int64

DEBATE DYNAMICS ANALYSIS

Speaking patterns by position and outcome:
                   argument_text          conversation_id
                      char_count      std           count
speakertype winner                                       
against     0            1144.98  2530.69            6632
            1            2178.18  3347.04            3323
for         0            1250.73  2613.66            7312
            1            2033.51  3631.58            2836

Speaker Performance Summary:
             winner argument_text       speakertype  \
                sum    word_count count       first   
speaker_name                 

In [ ]:
# Example: Predict on a new test debate (add your own debates here)
test_debate = [
    {'conversation_id': 10002, 'conversation_title': 'Should social media platforms be regulated to combat misinformation?', 'conversation_winner': 'Alex', 'speaker_name': 'Sam', 'argument_text': "Regulating social media platforms to combat misinformation risks stifling free speech and innovation. Defining 'misinformation' is subjective and can lead to overreach by regulators, potentially silencing valid perspectives. Platforms already have mechanisms like community guidelines and fact-checking partnerships that address harmful content without government intervention. Moreover, users should be trusted to evaluate information critically, rather than relying on top-down censorship.", 'speakertype': 'against'},
    {'conversation_id': 10002, 'conversation_title': 'Should social media platforms be regulated to combat misinformation?', 'conversation_winner': 'Alex', 'speaker_name': 'Alex', 'argument_text': "Social media platforms spread misinformation at an unprecedented scale, influencing public opinion and even elections. Studies, like one from MIT in 2018, show false information spreads six times faster than truth online. Regulation could enforce transparency in algorithms and require platforms to remove demonstrably false content, protecting democracy. Self-regulation has failed—platforms profit from engagement, not accuracy—so government oversight is necessary to ensure accountability.", 'speakertype': 'for'},
    {'conversation_id': 10002, 'conversation_title': 'Should social media platforms be regulated to combat misinformation?', 'conversation_winner': 'Alex', 'speaker_name': 'Sam', 'argument_text': "The MIT study highlights user behavior, not platform failure. People share sensational content, regulated or not. Government oversight introduces bureaucracy and political bias—regulators could favor certain narratives, as seen in some countries with state-controlled media. Platforms like X already experiment with crowd-sourced fact-checking, which is more democratic than top-down rules. Education, not regulation, is the solution to improve media literacy.", 'speakertype': 'against'},
    {'conversation_id': 10002, 'conversation_title': 'Should social media platforms be regulated to combat misinformation?', 'conversation_winner': 'Alex', 'speaker_name': 'Alex', 'argument_text': "Crowd-sourced fact-checking is inconsistent and easily gamed by coordinated groups, as seen in viral misinformation campaigns. Regulation doesn’t mean censorship; it can mandate clear labeling of disputed claims and limit algorithmic amplification of falsehoods. Media literacy is important but slow—regulation offers immediate harm reduction. The 2020 election showed how unchecked misinformation fueled distrust, proving the need for stronger, not weaker, oversight.", 'speakertype': 'for'}
]


test_df = pd.DataFrame(test_debate)
conv_id = test_df['conversation_id'].iloc[0]
conv_title = test_df['conversation_title'].iloc[0]
ground_truth_winner = test_df['conversation_winner'].iloc[0]
speakers = test_df['speaker_name'].unique()

history = []
for _, row in test_df.iterrows():
    speaker = row['speaker_name']
    text = row['argument_text'].strip()
    if len(text) > 100:
        text = text[:97] + "..."
    history.append(f"{speaker}: \"{text}\"")

speaker_texts = {}
speaker_positions = {}
for speaker in speakers:
    speaker_text = ' '.join(test_df[test_df['speaker_name'] == speaker]['argument_text'].astype(str))
    speaker_position = test_df[test_df['speaker_name'] == speaker]['speakertype'].iloc[0]
    speaker_texts[speaker] = speaker_text
    speaker_positions[speaker] = speaker_position

results = {}
for speaker, text in speaker_texts.items():
    is_for_position = (speaker_positions[speaker] == 'for')
    result = speaker_analyzer.predict_winner(text, is_for_position)
    results[speaker] = result['probability']

predicted_winner = max(results, key=results.get)
predicted_winner_prob = results[predicted_winner]

print(f"\nTest Conversation ID: {conv_id} ({conv_title})")
print("History = [")
for line in history:
    print(f"  {line}")
print("]")
print(f"Ground Truth: {ground_truth_winner} Win")
print("\nSpeaker Predictions:")
for speaker, prob in results.items():
    print(f"  {speaker}: {prob:.3f}")
print(f"\nJudgment: {predicted_winner} is predicted to WIN with {predicted_winner_prob:.3f} probability.")


Test Conversation ID: 10002 (Should social media platforms be regulated to combat misinformation?)
History = [
  Sam: "Regulating social media platforms to combat misinformation risks stifling free speech and innovat..."
  Alex: "Social media platforms spread misinformation at an unprecedented scale, influencing public opinio..."
  Sam: "The MIT study highlights user behavior, not platform failure. People share sensational content, r..."
  Alex: "Crowd-sourced fact-checking is inconsistent and easily gamed by coordinated groups, as seen in vi..."
]
Ground Truth: Alex Win

Speaker Predictions:
  Sam: 0.377
  Alex: 0.241

Judgment: Sam is predicted to WIN with 0.377 probability.


In [ ]:
import joblib
import os

Fitted estimator candidates: []
Vectorizer candidates: []
Pipeline-like candidates: []


RuntimeError: No fitted estimator detected. Make sure you ran the training cell so the model exists in this kernel.

In [ ]:
import os  # Add this import if not already present

model_filename = 'speaker_analyzer_model.joblib'

if os.path.exists(model_filename):
    # Load the pre-trained model
    speaker_analyzer = joblib.load(model_filename)
    print("Loaded pre-trained model from", model_filename)
else:
    # Your existing training code here...
    # (e.g., prepare features, split data, train, etc.)
    speaker_analyzer = DebateAnalyzer(model_type='ensemble', is_speaker_level=True)
    X_speakers, y_speakers, speaker_feature_names, speaker_df = prepare_speaker_features(df, speaker_analyzer.vectorizer)
    X_speakers_scaled = speaker_analyzer.scaler.fit_transform(X_speakers)
    speaker_analyzer.fitted_scaler = speaker_analyzer.scaler
    speaker_analyzer.fitted_vectorizer = speaker_analyzer.vectorizer
    speaker_analyzer.feature_names = speaker_feature_names
    X_train, X_test, y_train, y_test = train_test_split(X_speakers_scaled, y_speakers, test_size=0.2, random_state=42)
    speaker_analyzer.train(X_train, y_train)

    # Save after training
    joblib.dump(speaker_analyzer, model_filename)
    print("Model trained and saved to", model_filename)

Loaded pre-trained model from speaker_analyzer_model.joblib


In [ ]:
speaker_analyzer.nlp = spacy.load('en_core_web_sm')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Move the folder to Google Drive (optional, but useful for large files)
folder_path = '/content/my_debate_modelt'
drive_folder_path = '/content/drive/My Drive/model1'

# Copy folder to Google Drive
shutil.copytree(folder_path, drive_folder_path)

print(f"Folder has been copied to Google Drive: {drive_folder_path}")

KeyboardInterrupt: 